# Oscillating-field SABRE: Analytical model for a three-spin system
In this notebook, we will derive an analytical model for a three-spin system that is subjected to a oscillating polarisation transfer field. We consider a square-modulated field and derive a time-independent representation of the Hamiltonian using rotating frame transformations and average Hamiltonian theory. Finally, we show that the generalised LAC condition applies also for square waves and we calculate analytical expressions for the effective coupling.

The calculations are made using SymPy.

In [ ]:
from sympy import *

### Define symbols and spin operators
We will define all the necessary symbols and spin operators.

In [ ]:
ax, az0, az1 = symbols("a_x, a_z0, a_z1", real=True)
wp, t, tp = symbols(r"\omega_p, t, t'", positive=True, real=True)
az1f = Function("a_z1", real=True)(t)
m = symbols("m", integer=True)

Ix = Rational(1/2) * Matrix([
    [0, 1],
    [1, 0]
])

Iz = Rational(1/2) * Matrix([
    [1, 0],
    [0, -1]
])

### Write the time-dependent Hamiltonian
The Hamiltonian that describes the coherent spin dynamics that drive the transitions between $\ket{S\alpha}$ and $\ket{T_+\beta}$ (or $\ket{S\beta}$ and $\ket{T_-\alpha}$) can be reduced into a $2 \times 2$ block that contains the states of interest. This Hamiltonian is of form:
$$
\hat{H} = \frac{1}{2}\begin{bmatrix} a_{z0} + a_{z1}(t)  & a_x \\ a_x & -a_{z0} - a_{z1}(t) \end{bmatrix}
$$
The coefficients $a_x$, $a_{z0}$, and $a_{z1}(t)$ contain resonance frequencies $\omega_i$ for each spin $i$ and $J$-coupling strengths $J_{ij}$ between spins $i$ and $j$ and they are presented elsewhere.

In [ ]:
H = ax*Ix + (az0 + az1f)*Iz
display(H)

### Transform to the first rotating frame
We want to apply a rotating frame transformation to an interaction frame defined by the time-dependent term $a_{z1}(t)\hat{I}_z$. The unitary operator that achieves this transformation is:
$$
\hat{U}_1 = e^{i\hat{I}_z \int_{0}^{t} a_{z1}(t')dt'}
$$
The unitary transformation is applied to the Hamiltonian by:
$$
\tilde{H} = \hat{U}_1\hat{H}\hat{U}_1^{\dag} + i\frac{d\hat{U}_1}{dt}\hat{U}_1^{\dag}
$$

In [ ]:
U1 = exp(I*Iz*integrate(az1f.subs(t, tp), (tp, 0, t)))
display(U1)

Hr1 = U1 * H * U1.H + I*diff(U1, t)*U1.H
Hr1 = simplify(Hr1)
display(Hr1)

### Transform to the second rotating frame
We apply another rotating frame transformation where the interaction frame is chosen inspired by the Floquet theory. We define $m$ such that it corresponds to the Fourier mode for which $\left| a_{z0} + m\omega_p \right|$ is minimised. The unitary operator of interest is:
$$
\hat{U}_2 = e^{-i\hat{I}_z m\omega_p t}
$$
The transformation is applied again by:
$$
\tilde{\tilde{H}} = \hat{U}_2\tilde{H}\hat{U}_2^{\dag} + i\frac{d\hat{U}_2}{dt}\hat{U}_2^{\dag}
$$

In [ ]:
U2 = exp(-I*Iz*m*wp*t)
display(U2)

Hr2 = U2 * Hr1 * U2.H + I*diff(U2, t)*U2.H
Hr2 = simplify(Hr2)
display(Hr2)

### Consider the square wave
The above Hamiltonian is still exact formulation of the $2 \times 2$ representation. However, it is of our interest to derive a time-independent form of the Hamiltonian. We will choose to deal with the following square wave:
$$
a_{z1}(t) = 
\begin{cases} 
    a_{z1} & 0 \leq t < \pi/\omega_p \\
    - a_{z1} & \pi/\omega_p \leq t < 2\pi/\omega_p 
\end{cases}
$$
The Hamiltonian will be split into two parts that we call part A and B.

In [ ]:
Hr2A = simplify(Hr2.subs(
    Integral(az1f.subs(t, tp), (tp, 0, t)), Integral(az1, (tp, 0, t))
))
display(Hr2A)

Hr2B = simplify(Hr2.subs(
    Integral(az1f.subs(t, tp), (tp, 0, t)),
    Integral(az1, (tp, 0, pi/wp)) + Integral(-az1, (tp, pi/wp, t))
))
display(Hr2B)

### Apply the average Hamiltonian theory
We use the first-order average from the average Hamiltonian theory, which is:
$$
\tilde{\tilde{H}}^{(1)} = \frac{1}{2\pi/\omega_p} \int_{0}^{2\pi/\omega_p} \tilde{\tilde{H}} dt
$$

In [ ]:
Hr2A_eff = simplify(integrate(Hr2A, (t, 0, pi/wp)))
Hr2B_eff = simplify(integrate(Hr2B, (t, pi/wp, 2*pi/wp)))

Hr2_eff = (Hr2A_eff + Hr2B_eff) / (2*pi / wp)
Hr2_eff = simplify(Hr2_eff)
display(Hr2_eff)

### Interpreting the result
We find again that efficient polarisation transfer requires:
$$
a_{z0} + m\omega_p = 0
$$
This is the same as for sinusoidal magnetic field. Therefore, the LAC condition for the static offset field $B_0$ is the same.